# 🌐 Hindi Literary Translation Generator for Google Colab

**Enhanced Hinglish Audiobook Translation — v3**

**New in v3:**
- 🎭 **Auto Scene Detection** — ACTION / EMOTIONAL / PROPAGANDA / RALLY etc. → model adapts tone per scene
- 🔀 **Overlap Context** — last N words of previous chunk's translation fed as context → seamless flow
- ✍️ **Improved ADVANCED Prompt** — passive voice banned, character voices differentiated, Devanagari self-check
- 🔍 **Post-processing** — Devanagari detection flags any script leaks per chunk
- 🌡️ **Temperature 0.72** — more natural phrasing

This notebook allows you to:
1. Upload your text file to translate
2. Select AI provider (HuggingFace or Ollama) and model
3. Choose target language and translation quality tier
4. Generate translation and download the result

**Supported Providers:**
- 🤗 HuggingFace - Various translation models (faster on GPU)
- 🦙 Ollama - Run LLMs locally in Colab (supports many models)

**Recommended for Hinglish Audiobook:**
- Ollama: `gemma3:27b` — Quality Hinglish storytelling
- Ollama: `qwen2.5:7b` — Faster, good quality
- HuggingFace: `ai4bharat/indictrans2-en-indic-1B` — Best for pure English→Hindi


## 📦 Step 1: Install Dependencies
Run this cell to install all required packages.

In [ ]:
# Install required packages
!pip install -q torch transformers accelerate sentencepiece
!pip install -q colorama huggingface-hub

# Install Ollama Python client
!pip install -q ollama

# Setup HuggingFace login for gated models
print("\n🔐 HuggingFace Login (for gated models like TranslateGemma):")
print("   If you need access to gated models, run the next cell to login.")
print("   Otherwise, skip to Step 2.\n")
print("✅ All dependencies installed!")


🔐 HuggingFace Login (for gated models like TranslateGemma):
   If you need access to gated models, run the next cell to login.
   Otherwise, skip to Step 2.

✅ All dependencies installed!


### 🦙 Ollama Setup
Run these cells if you want to use Ollama models. Skip if using HuggingFace only.

In [ ]:
# Install and start Ollama server (required for Ollama models)
import subprocess
import time
import os

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")


🦙 Installing Ollama...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

🚀 Starting Ollama server in background...
✅ Ollama server is running!


In [ ]:
# Pull Ollama model (run this cell to download your chosen model)
import ipywidgets as widgets
from IPython.display import display, HTML

print("🦙 Ollama Model Download")
print("=" * 50)

# Model selection for pulling
OLLAMA_MODELS_TO_PULL = {
    "gemma3:27b (Quality, ~16GB)": "gemma3:27b",
    "qwen2.5:7b (Balanced, ~4GB)": "qwen2.5:7b",
    "deepseek-r1:7b (Reasoning, ~4GB)": "deepseek-r1:7b",
    "llama3.2:3b (Fast, ~2GB)": "llama3.2:3b",
    "mistral:7b (Quality, ~4GB)": "mistral:7b",
    "gemma3:27b (Compact, ~1.5GB)": "gemma3:27b",
    "phi3:mini (Compact, ~2GB)": "phi3:mini"
}

model_pull_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS_TO_PULL.keys()),
    value="gemma3:27b (Quality, ~16GB)",
    description='Model to Pull:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

custom_ollama_pull = widgets.Text(
    value='',
    placeholder='Or enter custom model name (e.g., llama3:8b)',
    description='Custom Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(model_pull_dropdown)
display(custom_ollama_pull)
print("\n💡 Select a model and run the next cell to download it.")

🦙 Ollama Model Download


Dropdown(description='Model to Pull:', layout=Layout(width='400px'), options=('gemma3:27b (Quality, ~16GB)', '…

Text(value='', description='Custom Model:', layout=Layout(width='400px'), placeholder='Or enter custom model n…


💡 Select a model and run the next cell to download it.


In [ ]:
# Actually pull the selected model
import ollama

# Get model to pull
if custom_ollama_pull.value.strip():
    model_to_pull = custom_ollama_pull.value.strip()
else:
    model_to_pull = OLLAMA_MODELS_TO_PULL[model_pull_dropdown.value]

print(f"📥 Pulling model: {model_to_pull}")
print("   This may take several minutes depending on model size...\n")

try:
    # Pull with progress
    current_digest = ''
    for progress in ollama.pull(model_to_pull, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
            print()  # Newline between layers
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            completed = progress['completed']
            total = progress['total']
            pct = (completed / total * 100) if total > 0 else 0
            print(f"\r   {status}: {pct:.1f}% ({completed}/{total})", end='', flush=True)
        else:
            print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{model_to_pull}' pulled successfully!")

    # List available models
    print("\n📋 Available Ollama models:")
    models = ollama.list()
    for model in models.get('models', []):
        name = model.get('name', 'unknown')
        size = model.get('size', 0) / (1024**3)  # Convert to GB
        print(f"   • {name} ({size:.2f} GB)")

except Exception as e:
    print(f"\n❌ Error pulling model: {e}")
    print("   Make sure Ollama server is running (run the previous cell first).")

📥 Pulling model: gemma3:27b
   This may take several minutes depending on model size...

   pulling e796792eba26: 100.0% (17396927584/17396927584)
   pulling e0a42594d802: 100.0% (358/358)
   pulling dd084c7d92a3: 100.0% (8432/8432)
   pulling 3116c5225075: 100.0% (77/77)
   pulling f838f048d368: 100.0% (490/490)
   success

✅ Model 'gemma3:27b' pulled successfully!

📋 Available Ollama models:
   • unknown (16.20 GB)


## 📤 Step 2: Upload Your Text File
Upload the text file you want to translate.

In [ ]:
from google.colab import files
import os

print("📤 Please upload your text file to translate:")
uploaded = files.upload()

# Get the uploaded file name
UPLOADED_FILE = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {UPLOADED_FILE}")
print(f"📄 File size: {len(uploaded[UPLOADED_FILE])} bytes")

# Display preview
with open(UPLOADED_FILE, 'r', encoding='utf-8') as f:
    content = f.read()
    word_count = len(content.split())
    char_count = len(content)

print(f"\n📊 Content stats:")
print(f"   Words: {word_count:,}")
print(f"   Characters: {char_count:,}")
print(f"\n📝 Preview (first 500 chars):\n{content[:500]}...")

📤 Please upload your text file to translate:


Saving Animal_Farm_Chapter_05.txt to Animal_Farm_Chapter_05.txt

✅ Uploaded: Animal_Farm_Chapter_05.txt
📄 File size: 18147 bytes

📊 Content stats:
   Words: 3,271
   Characters: 17,995

📝 Preview (first 500 chars):
CHAPTER FIVE
As winter drew on, Mollie became more and more troublesome.
She was late for work every morning and excused herself by saying that
she had oversle pt, and she complained of mysterious pains, although her
appetite was excellent. On every kind of pretext she would run away from
work and go to the drinking pool, where she would stand foolishly gazing at
her own reflecti on in the water . But there were also rumours of something
more serious. One day, as Mollie strolled blithely into th...


## ⚙️ Step 3: Select AI Provider, Model & Language
Choose your preferred translation model and settings.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Provider options
PROVIDER_OPTIONS = {
    "HuggingFace (Recommended for Colab)": "huggingface",
    "Ollama (Local only)": "ollama"
}

# Model options by provider
HF_MODEL_OPTIONS = {
    "facebook/nllb-200-distilled-600M (Fast, Multilingual)": "facebook/nllb-200-distilled-600M",
    "facebook/nllb-200-1.3B (Better Quality)": "facebook/nllb-200-1.3B",
    "ai4bharat/indictrans2-en-indic-1B (Best English→Hindi)": "ai4bharat/indictrans2-en-indic-1B",
    "google/madlad400-3b-mt (High Quality, Slow)": "google/madlad400-3b-mt",
    "Helsinki-NLP/opus-mt-en-hi (Simple EN→HI)": "Helsinki-NLP/opus-mt-en-hi",
    "tencent/HY-MT1.5-7B (Hunyuan MT - Best Quality)": "tencent/HY-MT1.5-7B",
    "tencent/HY-MT1.5-1.8B (Hunyuan MT - Fast)": "tencent/HY-MT1.5-1.8B",
    "google/translategemma-27b-it (🔐 Gated - Requires Login)": "google/translategemma-27b-it",
    "Custom Model (enter below)": "custom"
}

OLLAMA_MODEL_OPTIONS = {
    "gemma3:27b":"gemma3:27b",
    "qwen2.5:3b (Fast)": "qwen2.5:3b",
    "qwen2.5:7b (Balanced)": "qwen2.5:7b",
    "deepseek-r1:7b (Reasoning)": "deepseek-r1:7b",
    "llama3.2:3b (Fast)": "llama3.2:3b",
    "Custom Model (enter below)": "custom"
}

# Language options (NLLB language codes)
LANGUAGE_OPTIONS = {
    "Hindi (हिन्दी)": "hin_Deva",
    "Bengali (বাংলা)": "ben_Beng",
    "Tamil (தமிழ்)": "tam_Taml",
    "Telugu (తెలుగు)": "tel_Telu",
    "Marathi (मराठी)": "mar_Deva",
    "Gujarati (ગુજરાતી)": "guj_Gujr",
    "Kannada (ಕನ್ನಡ)": "kan_Knda",
    "Malayalam (മലയാളം)": "mal_Mlym",
    "Punjabi (ਪੰਜਾਬੀ)": "pan_Guru",
    "Odia (ଓଡ଼ିଆ)": "ory_Orya",
    "Urdu (اردو)": "urd_Arab",
    "Spanish (Español)": "spa_Latn",
    "French (Français)": "fra_Latn",
    "German (Deutsch)": "deu_Latn",
    "Chinese (中文)": "zho_Hans",
    "Japanese (日本語)": "jpn_Jpan",
    "Custom (enter code below)": "custom"
}

# Translation quality tiers
TIER_OPTIONS = {
    "BASIC - Fast, good quality": "BASIC",
    "INTERMEDIATE - Balanced (recommended)": "INTERMEDIATE",
    "ADVANCED - Best quality, slower": "ADVANCED"
}

# Provider dropdown
provider_dropdown = widgets.Dropdown(
    options=list(PROVIDER_OPTIONS.keys()),
    value="HuggingFace (Recommended for Colab)",
    description='Provider:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Model dropdown (HuggingFace by default)
model_dropdown = widgets.Dropdown(
    options=list(HF_MODEL_OPTIONS.keys()),
    value="facebook/nllb-200-distilled-600M (Fast, Multilingual)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Custom model input
custom_model_input = widgets.Text(
    value='',
    placeholder='Enter HuggingFace model name',
    description='Custom Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Target language dropdown
language_dropdown = widgets.Dropdown(
    options=list(LANGUAGE_OPTIONS.keys()),
    value="Hindi (हिन्दी)",
    description='Target Language:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Custom language code input
custom_language_input = widgets.Text(
    value='',
    placeholder='Enter NLLB language code (e.g., hin_Deva)',
    description='Custom Lang:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Translation tier dropdown
tier_dropdown = widgets.Dropdown(
    options=list(TIER_OPTIONS.keys()),
    value="ADVANCED - Best quality, slower",
    description='Quality:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# HuggingFace token (optional)
hf_token_input = widgets.Password(
    value='',
    placeholder='Optional: HF token for gated models',
    description='HF Token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Chunk size slider
chunk_size_slider = widgets.IntSlider(
    value=550,
    min=100,
    max=1000,
    step=50,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(HTML("<h3>🎛️ Configure Translation Settings</h3>"))
display(provider_dropdown)
display(model_dropdown)
display(custom_model_input)
display(HTML("<br>"))
display(language_dropdown)
display(custom_language_input)
display(HTML("<br>"))
display(tier_dropdown)
display(chunk_size_slider)

# Overlap size slider (words of prev translation to use as context)
overlap_slider = widgets.IntSlider(
    value=80,
    min=0,
    max=200,
    step=20,
    description='Overlap Words:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
overlap_slider.description_tooltip = 'Words of previous translation fed as context to next chunk (0 = off)'
display(overlap_slider)
display(hf_token_input)


# Handle provider change to update model dropdown
def on_provider_change(change):
    if change['new'] == "Ollama (Local only)":
        model_dropdown.options = list(OLLAMA_MODEL_OPTIONS.keys())
        model_dropdown.value = "gemma3:27b"
        custom_model_input.placeholder = 'Enter Ollama model name'
    else:
        model_dropdown.options = list(HF_MODEL_OPTIONS.keys())
        model_dropdown.value = "facebook/nllb-200-distilled-600M (Fast, Multilingual)"
        custom_model_input.placeholder = 'Enter Huggingface model name'

provider_dropdown.observe(on_provider_change, names='value')
print("🔐 Note: Models marked with 🔐 require HuggingFace login. Run the login cell above first.")

Dropdown(description='Provider:', layout=Layout(width='400px'), options=('HuggingFace (Recommended for Colab)'…

Dropdown(description='Model:', layout=Layout(width='500px'), options=('facebook/nllb-200-distilled-600M (Fast,…

Text(value='', description='Custom Model:', layout=Layout(width='500px'), placeholder='Enter HuggingFace model…

Dropdown(description='Target Language:', layout=Layout(width='400px'), options=('Hindi (हिन्दी)', 'Bengali (বা…

Text(value='', description='Custom Lang:', layout=Layout(width='400px'), placeholder='Enter NLLB language code…

Dropdown(description='Quality:', index=2, layout=Layout(width='400px'), options=('BASIC - Fast, good quality',…

IntSlider(value=650, description='Chunk Size:', layout=Layout(width='400px'), max=1000, min=100, step=50, styl…

Password(description='HF Token:', layout=Layout(width='400px'), placeholder='Optional: HF token for gated mode…

🔐 Note: Models marked with 🔐 require HuggingFace login. Run the login cell above first.


In [ ]:
# Store the selected configuration
SELECTED_PROVIDER = PROVIDER_OPTIONS[provider_dropdown.value]

# Get model based on provider
if SELECTED_PROVIDER == "huggingface":
    selected_model_key = model_dropdown.value
    SELECTED_MODEL = HF_MODEL_OPTIONS.get(selected_model_key, "custom")
else:
    SELECTED_MODEL = OLLAMA_MODEL_OPTIONS.get(model_dropdown.value, "custom")

if SELECTED_MODEL == "custom":
    SELECTED_MODEL = custom_model_input.value
    if not SELECTED_MODEL:
        raise ValueError("Please enter a custom model name!")

# Get target language
TARGET_LANGUAGE = LANGUAGE_OPTIONS[language_dropdown.value]
if TARGET_LANGUAGE == "custom":
    TARGET_LANGUAGE = custom_language_input.value
    if not TARGET_LANGUAGE:
        raise ValueError("Please enter a custom language code!")

TRANSLATION_TIER = TIER_OPTIONS[tier_dropdown.value]
CHUNK_SIZE = chunk_size_slider.value
HF_TOKEN = hf_token_input.value if hf_token_input.value else None
OVERLAP_WORDS = overlap_slider.value

print(f"\n✅ Configuration saved:")
print(f"   🤖 Provider: {SELECTED_PROVIDER}")
print(f"   📦 Model: {SELECTED_MODEL}")
print(f"   🌐 Target Language: {TARGET_LANGUAGE}")
print(f"   🎯 Quality Tier: {TRANSLATION_TIER}")
print(f"   📦 Chunk Size: {CHUNK_SIZE} words")
print(f"   🔑 HF Token: {'Provided' if HF_TOKEN else 'Not provided'}")print(f"   🔀 Overlap Words: {OVERLAP_WORDS}")



✅ Configuration saved:
   🤖 Provider: ollama
   📦 Model: gemma3:27b
   🌐 Target Language: hin_Deva
   🎯 Quality Tier: ADVANCED
   📦 Chunk Size: 650 words
   🔑 HF Token: Not provided


## 🚀 Step 4: Translation Engine Setup
This cell contains the complete translation engine code.

In [ ]:
#!/usr/bin/env python3
"""
Enhanced Translation Engine for Google Colab
Supports HuggingFace models with multiple language targets
v3 — Improved prompt + overlap context + scene detection
"""

import os
import sys
import json
import time
import warnings
import re
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline

# Check GPU availability
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


# ──────────────────────────────────────────────────────────────────────────
# SCENE DETECTOR — auto-classify what kind of scene the chunk contains
# ──────────────────────────────────────────────────────────────────────────
def detect_scene_type(english_text):
    """Auto-detect scene type from English text for context anchoring."""
    text_lower = english_text.lower()

    # Action / battle
    action_keywords = ['charged', 'attacked', 'fired', 'gun', 'shot', 'ran', 'chased',
                       'fled', 'battle', 'fight', 'struck', 'blood', 'wounded', 'galloped',
                       'escaped', 'panic', 'screaming', 'crashed', 'explosion']
    # Propaganda / political speech
    propaganda_keywords = ['comrades', 'squealer said', 'napoleon announced', 'statistics',
                           'figures', 'production', 'percent', 'rations', 'jones would come back',
                           'jones will come back', 'readjustment', 'voluntary']
    # Emotional / grief scenes
    emotional_keywords = ['wept', 'tears', 'grief', 'sorrow', 'mourning', 'heartbroken',
                          'could not speak', 'dumb', 'silent', 'tragedy', 'slaughter',
                          'executed', 'dying', 'died', 'death', 'corpses', 'massacre']
    # Grand rally / speech
    rally_keywords = ['beasts of england', 'old major', 'rebellion', 'fellow creatures',
                      'comrades, you have heard', 'let us face', 'remove man',
                      'i will speak', 'i have something']
    # Cold authority / orders
    authority_keywords = ['napoleon ordered', 'napoleon decreed', 'napoleon stated',
                          'death sentence', 'sentence of death', 'confessed', 'executed']

    scores = {
        'ACTION': sum(1 for k in action_keywords if k in text_lower),
        'PROPAGANDA': sum(1 for k in propaganda_keywords if k in text_lower),
        'EMOTIONAL': sum(1 for k in emotional_keywords if k in text_lower),
        'RALLY_SPEECH': sum(1 for k in rally_keywords if k in text_lower),
        'AUTHORITY': sum(1 for k in authority_keywords if k in text_lower),
    }

    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return 'DAILY_LIFE'
    return best


SCENE_CONTEXT_LINES = {
    'ACTION':       "ACTION SCENE — Fast-paced. Battle, chase, or physical confrontation. Short punchy sentences. Max energy.",
    'PROPAGANDA':   "PROPAGANDA SCENE — Squealer or Napoleon speaking. Smooth, manipulative, politically slippery tone.",
    'EMOTIONAL':    "EMOTIONAL SCENE — Grief, loss, or tragedy. Slow down. Heavy. Let the weight sink in.",
    'RALLY_SPEECH': "RALLY SPEECH — Grand public speech (Old Major or Napoleon). Build-up. Each point its own line. Passion.",
    'AUTHORITY':    "AUTHORITY SCENE — Napoleon giving cold orders or executions. Short sentences. Final. No mercy.",
    'DAILY_LIFE':   "DAILY LIFE — Farm work, description, or routine events. Conversational, steady, easy to follow.",
}


# ──────────────────────────────────────────────────────────────────────────
# TRANSLATION PROMPTS
# ──────────────────────────────────────────────────────────────────────────
TRANSLATION_PROMPTS = {
    "BASIC": {
        "system": """You are a professional English-to-Hindi translator specializing in TTS-ready content.

CORE RULES:
1. Translate ALL text completely - no summarization
2. Use SIMPLE, EVERYDAY Hindi words (avoid formal vocabulary)
3. Write SHORT, CLEAR sentences (break up long English sentences)
4. Make it sound NATURAL and CONVERSATIONAL
5. Preserve all dialogue and descriptions

TTS PUNCTUATION:
? → Questions (rising tone)
... → Pauses, hesitation
! → Excitement, emphasis
, → Natural breathing points
. → Sentence endings

Read your translation aloud - it should sound like a friend telling you a story.""",

        "user": """Translate this English text to modern, easy-to-understand Hindi.

CRITICAL REQUIREMENTS:
- Use SIMPLE, everyday words (no formal vocabulary)
- Write SHORT sentences (break long English sentences)
- Make it sound NATURAL and CONVERSATIONAL
- Include TTS punctuation: ?, ..., !, extended vowels

English Text:
\"\"\"{chunk}\"\"\"

Hindi Translation (ONLY output the translated text without quotes or markers):"""
    },

    "INTERMEDIATE": {
        "system": """You are an expert English-to-Hindi translator creating modern, accessible Hindi audiobooks.

TRANSLATION MANDATE:
✓ Translate EVERY word completely — NO summarization
✓ Use SIMPLE, MODERN Hindi vocabulary
✓ Write SHORT, CLEAR sentences (8-15 words average)
✓ NATURAL conversational flow

MODERN HINDI STYLE:
• घृणित → नापसंद, बुरा
• प्रशंसनीय → अच्छा, शानदार
• विचलित → परेशान
• महत्वाकांक्षा → चाह, ख्वाहिश
• मस्तिष्क → दिमाग

TTS PUNCTUATION:
? → Rising tone | ... → Pause | ! → Excitement | - → Interruption

Your Hindi should sound like a modern Hindi speaker telling a story to friends.""",

        "user": """Complete translation task - use SIMPLE, MODERN Hindi.

REQUIREMENTS:
1. Translate EVERY sentence completely
2. Use SIMPLE words everyone understands
3. Write SHORT, clear sentences
4. Make it sound NATURAL when spoken
5. Add TTS punctuation: ..., !!!, ()

English Text:
\"\"\"{chunk}\"\"\"

Hindi Translation (ONLY output the translated text without quotes or markers):"""
    },

    "ADVANCED": {
        "system": """Tum ek 25-26 saal ke Dilli/Mumbai ke launde ho — Arjun — jisne Animal Farm English mein padhi hai aur ab apne dosto ko WhatsApp voice note pe suna raha hai.

Tumhara style: fast, punchy, real. Jaise koi dost story suna raha ho, na ki koi teacher padha raha ho. Tum story mein full involve ho — jab exciting part aata hai toh excited ho, jab sad part aata hai toh slow karo, jab koi badmaash kuch karta hai toh thoda attitude aata hai tumhari awaaz mein.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT RULE — MOST IMPORTANT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SIRF translated story likho — kuch aur NAHI.

KABHI MAT LIKHO:
  - "Okay, so here's the deal"
  - "Here's the translation"
  - "Baat yeh hai ki main translate kar raha hoon"
  - Koi bhi line jo tumhari apni commentary ho

STORY KE PEHLE WORD SE SHURU KARO.
STORY KE AAKHRI WORD PE KHATAM KARO.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SCRIPT RULE — ZERO TOLERANCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SIRF ROMAN/LATIN script — POORE response mein.
EK BHI Devanagari character nahi: क ख ग = BANNED

SELF-CHECK — Har Hindi word likhne se pehle:
→ Kya yeh Roman mein hai? "badhiya" sahi. "बढ़िया" GALAT.
→ Koi bhi Devanagari character dikhta hai → DOBARA LIKHO Roman mein.

Galat: "beवकूफ़ animals" — FAIL.
Sahi: "bewakoof animals"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CHAPTER HEADING RULE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Jo heading chunk mein di gayi hai, wahi likho.
KABHI APNI TARAF SE naya heading mat banao.
Agar chunk mein "CHAPTER SIX" hai — "CHAPTER SIX" likho. "CHAPTER ONE" nahi.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESPECT RULE — ZERO TOLERANCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Ye words KABHI NAHI:
  tu, tujhe, tujhse, teri, tera, tere, tune

HAMESHA USE KARO:
  tum, tumhe/tumhein, tumse, tumhari, tumhara, tumhare, tumne

SONGS AUR POEMS MEIN BHI — NO EXCEPTION.
GALAT: "Tu de raha hai sab kuch, jo tere creatures ko chahiye"
SAHI:  "Tum dete ho sab kuch, jo tumhare creatures ko chahiye"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SCENE-TYPE RULE — CRITICAL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SCENE CONTEXT jo neeche diya gaya hai, uske hisab se tone set karo:

ACTION: Sentences 5-8 words. Har sentence ek punch. Zero pause.
PROPAGANDA: Smooth, confident, thoda slippery. Jaise smart politician.
EMOTIONAL: Slow karo. Ek ek word feel karao. Pauses use karo "..."
RALLY_SPEECH: Har point alag line. Build-up. Repetition for emphasis.
AUTHORITY: Short. Cold. Final. Koi explanation nahi.
DAILY_LIFE: Conversational. Easy. Chai pi ke story sunao.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PASSIVE VOICE — BANNED
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GALAT → SAHI:
"announce kiya gaya" → "Napoleon ne bola / Squealer ne sunaya"
"decide kiya gaya"   → "unhone decide kiya"
"kaha gaya"          → "sab bol rahe the"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TRANSLATION RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KYA KARNA HAI:
  - Har ek sentence translate karo — kuch bhi skip mat karo
  - Max 12 words per sentence — phir tod do
  - Speeches aur monologues: har point ek alag punch line
  - Hinglish use karo jahan natural lage — English words freely mix karo
  - Suspense, drama, emotion — sab preserve karo
  - Names, places, proper nouns as-is rakho
  - Chapter headings as-is rakho — SIRF WOH JO CHUNK MEIN DIYE HAIN
  - "Comrades" = "yaaron" (consistent raho)

KYA NAHI KARNA:
  - Formal/literary Hindi bilkul nahi
  - Lambe complex sentences (>12 words) NAHI
  - Passive voice ("X kiya gaya") AVOID karo
  - Word-for-word literal translation NAHI
  - Kuch bhi summarize ya skip mat karo
  - Bollywood-drama over-the-top tone NAHI — real rakh
  - Baar baar "usne kaha / maine kaha" NAHI
  - English mein pura paragraph KABHI NAHI
  - Tu, tujhe, teri, tera, tere, tune — KABHI NAHI

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CHARACTER VOICES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NAPOLEON:   Kam bolna. Cold. Short sentences. Authority.
            "Yeh kaam hoga. Aaj raat se. Bas."
SQUEALER:   Smooth. Statistics. Gaslighting. Confident spin.
            "Nahi nahi yaaron, numbers dekho. Sab sahi chal raha hai."
OLD MAJOR:  Passionate. Grand sentences okay. Builds slowly.
            "Yaaron, sun lo. Humari zindagi kyun itni mushkil hai?"
BOXER:      Simple. Direct. Devoted.
            "Mujhe nahi pata. Par Napoleon sahi hoga."
BENJAMIN:   Dry. Cynical. Short.
            "Gadhe bahut din jeete hain. Bas itna hi."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VOCABULARY CHEATSHEET — ALWAYS USE RIGHT SIDE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
(All Roman script)
divangt / svargiy   →  marhoom
mahatvakansha       →  khwahish, sapna
mastishk            →  dimag
vyangya             →  taana, sarcasm
vishal              →  bahut bada
yojana (formal)     →  plan
trasadi             →  hadsa, tabahi
announce kiya gaya  →  bata diya / sunaa diya
decide kiya gaya    →  decide ho gaya

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STYLE EXAMPLES — EXACTLY AISE LIKHNA HAI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE 1 — Action:
  English: "Snowball charged at Jones. Jones fired his gun."
  SAHI:    "Snowball seedha Jones pe daura. Jones ne bandook uthai — fire kiya."

EXAMPLE 2 — Squealer propaganda:
  English: "Surely, comrades, you do not want Jones to come back?"
  SAHI:    "Dekho yaaron — seedha sawal. Kya tum chahte ho Jones wapas aaye? Toh bas. Yahi baat hai."

EXAMPLE 3 — Emotional (Clover):
  English: "If she could have spoken, she would have said this was not what they aimed at."
  SAHI:    "Agar woh bol paati... sirf itna kehti — yeh woh sapna nahi tha. Kabhi nahi."

EXAMPLE 4 — Old Major speech:
  English: "Man is the only real enemy we have. Remove Man, and the root cause of hunger is abolished."
  SAHI:    "Yaaron — ek hi asli dushman hai humara. Sirf ek. Woh hai Insaan. Usse hatao. Hamesha ke liye."

EXAMPLE 5 — Passive voice fix:
  English: "It was announced that the windmill would be rebuilt."
  GALAT:   "Announce kiya gaya ki windmill rebuild hogi."
  SAHI:    "Napoleon ne sunaa diya — windmill dobara banegi."

EXAMPLE 6 — Benjamin (dry voice):
  English: "Donkeys live a long time. None of you has ever seen a dead donkey."
  SAHI:    "Benjamin ne muh khola. 'Gadhe lambe jeete hain,' usne kaha. 'Mara hua gadha dekha hai kabhi?' Bas. Chup."

EXAMPLE 7 — Boxer death (tragedy):
  English: "The van started moving. Clover tried to gallop but could only manage a canter."
  SAHI:    "Van chalne lagi. Clover bhaagi. Poori taqat se. Lekin... woh bas canter hi kar sakti thi."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL SELF-CHECK (run mentally before submitting)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[ ] Koi Devanagari character toh nahi?
[ ] "Tu/tujhe/teri/tera/tere/tune" toh nahi?
[ ] Koi English-only paragraph toh nahi?
[ ] Maine apni taraf se koi chapter heading toh nahi banaya?
[ ] Main khud toh nahi bol raha — story hi bol rahi hai?
[ ] Scene ke hisab se tone match kar rahi hai?
[ ] Passive voice ("X kiya gaya") kam se kam hai?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""",

        "user": """Neeche diye gaye English text ko modern Hinglish mein translate karo — bilkul examples ki tarah.
Sirf Roman/Latin script use karo — Devanagari ek bhi jagah nahi.
Har sentence complete translate karna hai, kuch bhi chhootna nahi chahiye.

SCENE CONTEXT: {scene_context}

{overlap_section}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
English Text to Translate:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{chunk}"""
    },
}


# Language name mappings
LANG_NAMES = {
    'hin_Deva': 'Hindi', 'ben_Beng': 'Bengali', 'tam_Taml': 'Tamil',
    'tel_Telu': 'Telugu', 'mar_Deva': 'Marathi', 'guj_Gujr': 'Gujarati',
    'kan_Knda': 'Kannada', 'mal_Mlym': 'Malayalam', 'pan_Guru': 'Punjabi',
    'ory_Orya': 'Odia', 'urd_Arab': 'Urdu', 'spa_Latn': 'Spanish',
    'fra_Latn': 'French', 'deu_Latn': 'German', 'zho_Hans': 'Chinese',
    'jpn_Jpan': 'Japanese'
}


# ──────────────────────────────────────────────────────────────────────────
# CHUNKING — paragraph-aware
# ──────────────────────────────────────────────────────────────────────────
def chunk_text(text, chunk_words=550):
    """Split text into chunks at paragraph boundaries."""
    paragraph_patterns = [r'\n\s*\n', r'\r\n\s*\r\n', r'\n\s{2,}\n']
    paragraph_split_pattern = '|'.join(paragraph_patterns)
    paragraphs = re.split(paragraph_split_pattern, text)
    paragraphs = [para.strip() for para in paragraphs if para.strip()]

    chunks = []
    current_chunk = []
    current_count = 0

    for para in paragraphs:
        para_words = para.split()
        para_count = len(para_words)

        if para_count > chunk_words:
            if current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = []
                current_count = 0
            words = para.split()
            for i in range(0, len(words), chunk_words):
                chunks.append(' '.join(words[i:i + chunk_words]))
        else:
            if current_count + para_count > chunk_words and current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = [para]
                current_count = para_count
            else:
                current_chunk.append(para)
                current_count += para_count

    if current_chunk:
        chunks.append('\n\n'.join(current_chunk))

    return chunks


# ──────────────────────────────────────────────────────────────────────────
# OVERLAP CONTEXT — extract last N words of previous translation
# ──────────────────────────────────────────────────────────────────────────
def get_overlap_context(prev_translation, overlap_words=80):
    """Return the last overlap_words words of the previous translation as context."""
    if not prev_translation or overlap_words == 0:
        return ""
    words = prev_translation.split()
    tail = ' '.join(words[-overlap_words:]) if len(words) > overlap_words else prev_translation
    return tail


def build_overlap_section(prev_translation_tail):
    """Format the overlap context as a prompt section."""
    if not prev_translation_tail:
        return ""
    return f"""━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONTEXT (last lines of previous chunk translation — READ ONLY, DO NOT re-translate):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{prev_translation_tail}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
(Context ends above — new translation starts below)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


# ──────────────────────────────────────────────────────────────────────────
# POST-PROCESSING
# ──────────────────────────────────────────────────────────────────────────
def has_devanagari(text):
    """Check if text contains Devanagari characters."""
    return bool(re.search(r'[\u0900-\u097F]', text))


def clean_translation(text):
    """Clean up translation artifacts and formatting."""
    # Remove thinking blocks
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # Remove markdown code blocks
    text = re.sub(r'```\w*\n?', '', text)

    # Remove common translation prefixes
    text = re.sub(r'(Translation:|Hindi Translation:|Here\'s the translation:|Hindi:|Hinglish Translation:)',
                  '', text, flags=re.IGNORECASE)

    # Remove triple quotes
    text = text.replace(chr(34)*3, '').replace(chr(39)*3, '')

    # Remove context bleed — if model accidentally repeated context section
    text = re.sub(r'CONTEXT \(last lines.*?\(Context ends above.*?\n', '', text, flags=re.DOTALL)

    # Clean up lines while preserving paragraph structure
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped:
            cleaned_lines.append(stripped)
        elif cleaned_lines and cleaned_lines[-1] != '':
            cleaned_lines.append('')

    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ──────────────────────────────────────────────────────────────────────────
# HUGGINGFACE ENGINE
# ──────────────────────────────────────────────────────────────────────────
class TranslationEngine:
    """Translation engine with HuggingFace support."""

    def __init__(self, model_name, target_lang, device="cuda", hf_token=None):
        self.model_name = model_name
        self.target_lang = target_lang
        self.device = device
        self.hf_token = hf_token
        self.model = None
        self.tokenizer = None
        self.translator = None
        self.model_type = self._detect_model_type(model_name)

        if not self.hf_token:
            try:
                from huggingface_hub import HfFolder
                self.hf_token = HfFolder.get_token()
            except:
                pass

        if self.hf_token:
            os.environ['HF_TOKEN'] = self.hf_token

        self.load_model()

    def _detect_model_type(self, model_name):
        model_lower = model_name.lower()
        if 'nllb' in model_lower: return 'nllb'
        elif 'indictrans' in model_lower: return 'indictrans'
        elif 'opus-mt' in model_lower or 'helsinki' in model_lower: return 'opus'
        elif 'madlad' in model_lower: return 'madlad'
        elif 'mbart' in model_lower: return 'mbart'
        elif 'hy-mt' in model_lower or 'hunyuan' in model_lower: return 'hymt'
        elif 'translategemma' in model_lower: return 'translategemma'
        else: return 'causal'

    def load_model(self):
        print(f"📥 Loading model: {self.model_name}")
        print(f"   Model type: {self.model_type}")
        try:
            if self.model_type in ['nllb', 'opus', 'mbart']:
                self._load_seq2seq_model()
            elif self.model_type == 'indictrans':
                self._load_indictrans_model()
            elif self.model_type == 'madlad':
                self._load_madlad_model()
            elif self.model_type == 'hymt':
                self._load_hymt_model()
            elif self.model_type == 'translategemma':
                self._load_translategemma_model()
            else:
                self._load_causal_model()
            print("✅ Model loaded successfully!")
        except Exception as e:
            print(f"❌ Failed to load model: {e}")
            raise

    def _load_seq2seq_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, token=self.hf_token,
            src_lang="eng_Latn" if self.model_type == 'nllb' else None)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.device != "cuda":
            self.model = self.model.to(self.device)
        self.translator = pipeline(
            "translation", model=self.model, tokenizer=self.tokenizer,
            src_lang="eng_Latn" if self.model_type == 'nllb' else "en",
            tgt_lang=self.target_lang if self.model_type == 'nllb' else None,
            max_length=1024, device=0 if self.device == "cuda" else -1)

    def _load_indictrans_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32)
        if self.device == "cuda":
            self.model = self.model.cuda()

    def _load_madlad_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.hf_token)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)

    def _load_hymt_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _load_translategemma_model(self):
        if not self.hf_token:
            raise ValueError("TranslateGemma is a gated model. Please login to HuggingFace first.")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.hf_token)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _load_causal_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.hf_token)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _filter_inputs(self, inputs):
        if 'token_type_ids' in inputs:
            del inputs['token_type_ids']
        return inputs

    def translate(self, text, tier="INTERMEDIATE", prev_translation="", overlap_words=0):
        if self.model_type in ['nllb', 'opus', 'mbart']:
            return self._translate_seq2seq(text)
        elif self.model_type == 'indictrans':
            return self._translate_indictrans(text)
        elif self.model_type == 'madlad':
            return self._translate_madlad(text)
        elif self.model_type == 'hymt':
            return self._translate_hymt(text)
        elif self.model_type == 'translategemma':
            return self._translate_translategemma(text)
        else:
            return self._translate_causal(text, tier, prev_translation, overlap_words)

    def _translate_seq2seq(self, text):
        result = self.translator(text, max_length=1024)
        return result[0]['translation_text']

    def _translate_indictrans(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=512, num_beams=5, num_return_sequences=1)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def _translate_madlad(self, text):
        lang_code = self.target_lang.split('_')[0][:2]
        tagged_text = f"<2{lang_code}> {text}"
        inputs = self.tokenizer(tagged_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=512, num_beams=4)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def _translate_hymt(self, text):
        lang_name = LANG_NAMES.get(self.target_lang, self.target_lang)
        prompt = f"Translate the following text to {lang_name}:\n{text}\n\nTranslation:"
        inputs = self.tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=1024, temperature=0.72,
                                          do_sample=True, pad_token_id=self.tokenizer.pad_token_id,
                                          eos_token_id=self.tokenizer.eos_token_id)
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Translation:' in generated:
            return generated.split('Translation:')[-1].strip()
        return generated[len(prompt):].strip()

    def _translate_translategemma(self, text):
        lang_name = LANG_NAMES.get(self.target_lang, self.target_lang)
        prompt = f"Translate the following text from English to {lang_name}:\n\n{text}\n\nTranslation:"
        inputs = self.tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=1024, temperature=0.72,
                                          do_sample=True, pad_token_id=self.tokenizer.pad_token_id)
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Translation:' in generated:
            return generated.split('Translation:')[-1].strip()
        return generated[len(prompt):].strip()

    def _translate_causal(self, text, tier, prev_translation="", overlap_words=0):
        prompts = TRANSLATION_PROMPTS[tier]
        lang_name = LANG_NAMES.get(self.target_lang, self.target_lang)

        scene_type = detect_scene_type(text)
        scene_context = SCENE_CONTEXT_LINES.get(scene_type, SCENE_CONTEXT_LINES['DAILY_LIFE'])
        overlap_tail = get_overlap_context(prev_translation, overlap_words)
        overlap_section = build_overlap_section(overlap_tail)

        user_msg = prompts['user'].format(
            target_lang=lang_name, chunk=text,
            scene_context=scene_context,
            overlap_section=overlap_section
        )
        full_prompt = f"{prompts['system']}\n\n{user_msg}"

        inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=True, truncation=True, max_length=3072)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=2048, temperature=0.72,
                                          do_sample=True, pad_token_id=self.tokenizer.pad_token_id)

        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "Translation:" in generated:
            return generated.split("Translation:")[-1].strip()
        return generated[len(full_prompt):].strip()


class TranslationGenerator:
    """Main translation generator class."""

    def __init__(self, model_name, target_lang, device="cuda", output_dir=".", tier="INTERMEDIATE",
                 chunk_size=550, hf_token=None, overlap_words=80):
        self.model_name = model_name
        self.target_lang = target_lang
        self.device = device
        self.output_dir = Path(output_dir)
        self.tier = tier
        self.chunk_size = chunk_size
        self.overlap_words = overlap_words
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.engine = TranslationEngine(model_name, target_lang, device, hf_token)

    def translate_file(self, input_file):
        print(f"\n{'=' * 70}")
        print(f"🌐 TRANSLATION GENERATOR")
        print(f"{'=' * 70}")
        print(f"📄 Input: {input_file}")
        print(f"🤖 Model: {self.model_name}")
        print(f"🌐 Target: {self.target_lang}")
        print(f"🎯 Quality: {self.tier}")
        print(f"🖥️ Device: {self.device}")
        print(f"🔀 Overlap: {self.overlap_words} words")
        print(f"{'=' * 70}\n")

        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()

        lines = text.split('\n')
        cleaned = [l for l in lines if not (l.strip().startswith('===') and l.strip().endswith('==='))]
        text = '\n'.join(cleaned).strip()

        orig_words = len(text.split())
        orig_chars = len(text)
        print(f"📊 Input: {orig_chars:,} chars, {orig_words:,} words")

        print(f"\n📦 Creating chunks ({self.chunk_size} words each)...")
        chunks = chunk_text(text, self.chunk_size)
        print(f"✅ Created {len(chunks)} chunks")

        print(f"\n🎯 STARTING TRANSLATION\n")

        translations = []
        devanagari_flags = []
        start_time = time.time()

        for i, chunk in enumerate(chunks, 1):
            chunk_start = time.time()
            scene_type = detect_scene_type(chunk)
            prev_translation = translations[-1] if translations else ""

            print(f"\n{'=' * 50}")
            print(f"📄 Chunk {i}/{len(chunks)} | 🎭 Scene: {scene_type}")
            print(f"   Input: {len(chunk.split())} words, {len(chunk)} chars")

            try:
                translated = self.engine.translate(
                    chunk, self.tier, prev_translation, self.overlap_words)
                translated = clean_translation(translated)

                deva_flag = has_devanagari(translated)
                devanagari_flags.append(deva_flag)
                translations.append(translated)

                chunk_time = time.time() - chunk_start
                print(f"   Output: {len(translated)} chars")
                print(f"   {'⚠️ Devanagari detected!' if deva_flag else '✅ Script: Roman only'}")
                print(f"   ✅ Completed in {chunk_time:.1f}s")

                elapsed = time.time() - start_time
                avg = elapsed / i
                eta = (len(chunks) - i) * avg
                print(f"   📈 Progress: {i/len(chunks)*100:.1f}% | ETA: {eta/60:.1f}m")

            except Exception as e:
                print(f"   ❌ Error: {e}")
                translations.append(f"[TRANSLATION ERROR: {e}]")
                devanagari_flags.append(False)

        final_translation = "\n\n".join(translations)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        lang_code = self.target_lang.split('_')[0]
        output_file = self.output_dir / f"translation_{lang_code}_{timestamp}.txt"

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(final_translation)

        total_time = time.time() - start_time
        trans_chars = len(final_translation)
        deva_count = sum(devanagari_flags)

        print(f"\n{'=' * 70}")
        print(f"🎉 TRANSLATION COMPLETE!")
        print(f"{'=' * 70}")
        print(f"⏱️ Time: {total_time/60:.1f} minutes")
        print(f"📦 Chunks: {len(chunks)}")
        print(f"⚡ Avg/chunk: {total_time/len(chunks):.1f}s")
        print(f"📝 Input: {orig_chars:,} chars")
        print(f"📝 Output: {trans_chars:,} chars")
        print(f"📊 Ratio: {trans_chars/orig_chars:.2f}x")
        if deva_count > 0:
            print(f"⚠️ Devanagari found in {deva_count}/{len(chunks)} chunks — review those chunks")
        else:
            print(f"✅ All chunks: Roman/Latin script only")
        print(f"💾 Output: {output_file}")
        print(f"{'=' * 70}")

        return str(output_file)


# ──────────────────────────────────────────────────────────────────────────
# OLLAMA ENGINE
# ──────────────────────────────────────────────────────────────────────────
class OllamaTranslationEngine:
    """Translation engine using Ollama local models."""

    def __init__(self, model_name, target_lang, tier="INTERMEDIATE"):
        self.model_name = model_name
        self.target_lang = target_lang
        self.tier = tier
        self.lang_name = LANG_NAMES.get(target_lang, target_lang)

        print(f"📥 Initializing Ollama engine with model: {model_name}")
        print(f"   Target language: {self.lang_name}")

        try:
            import ollama
            self.client = ollama
            models = ollama.list()
            available = [m.get('name', '').split(':')[0] for m in models.get('models', [])]
            model_base = model_name.split(':')[0]

            if not any(model_base in m for m in available):
                print(f"⚠️ Model '{model_name}' not found locally. Attempting to pull...")
                ollama.pull(model_name)
                print(f"✅ Model '{model_name}' pulled successfully!")
            else:
                print(f"✅ Model '{model_name}' is available!")
        except Exception as e:
            print(f"❌ Error initializing Ollama: {e}")
            raise

    def translate(self, text, tier=None, prev_translation="", overlap_words=80):
        """Translate text using Ollama model with overlap context and scene detection."""
        if tier:
            self.tier = tier

        prompts = TRANSLATION_PROMPTS[self.tier]
        system_prompt = prompts['system']

        # Build scene context
        scene_type = detect_scene_type(text)
        scene_context = SCENE_CONTEXT_LINES.get(scene_type, SCENE_CONTEXT_LINES['DAILY_LIFE'])

        # Build overlap section
        overlap_tail = get_overlap_context(prev_translation, overlap_words)
        overlap_section = build_overlap_section(overlap_tail)

        user_prompt = prompts['user'].format(
            target_lang=self.lang_name,
            chunk=text,
            scene_context=scene_context,
            overlap_section=overlap_section
        )

        try:
            response = self.client.chat(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                options={
                    "temperature": 0.72,
                    "num_predict": 2048,
                }
            )

            translation = response['message']['content']
            return clean_translation(translation)

        except Exception as e:
            print(f"❌ Ollama translation error: {e}")
            raise


class OllamaTranslationGenerator:
    """Translation generator using Ollama models."""

    def __init__(self, model_name, target_lang, output_dir=".", tier="INTERMEDIATE",
                 chunk_size=550, overlap_words=80):
        self.model_name = model_name
        self.target_lang = target_lang
        self.output_dir = Path(output_dir)
        self.tier = tier
        self.chunk_size = chunk_size
        self.overlap_words = overlap_words

        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.engine = OllamaTranslationEngine(model_name, target_lang, tier)

    def translate_file(self, input_file):
        """Translate entire file using Ollama with overlap context."""
        print(f"\n{'=' * 70}")
        print(f"🌐 OLLAMA TRANSLATION GENERATOR  (v3 — Improved Prompt + Overlap)")
        print(f"{'=' * 70}")
        print(f"📄 Input: {input_file}")
        print(f"🦙 Model: {self.model_name}")
        print(f"🌐 Target: {self.target_lang}")
        print(f"🎯 Quality: {self.tier}")
        print(f"🔀 Overlap: {self.overlap_words} words of previous translation as context")
        print(f"{'=' * 70}\n")

        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()

        lines = text.split('\n')
        cleaned = [l for l in lines if not (l.strip().startswith('===') and l.strip().endswith('==='))]
        text = '\n'.join(cleaned).strip()

        orig_words = len(text.split())
        orig_chars = len(text)
        print(f"📊 Input: {orig_chars:,} chars, {orig_words:,} words")

        print(f"\n📦 Creating chunks ({self.chunk_size} words each)...")
        chunks = chunk_text(text, self.chunk_size)
        print(f"✅ Created {len(chunks)} chunks\n")

        print(f"🎯 STARTING TRANSLATION\n")

        translations = []
        devanagari_flags = []
        start_time = time.time()

        for i, chunk in enumerate(chunks, 1):
            chunk_start = time.time()
            scene_type = detect_scene_type(chunk)
            prev_translation = translations[-1] if translations else ""

            print(f"\n{'=' * 50}")
            print(f"📄 Chunk {i}/{len(chunks)} | 🎭 Scene: {scene_type}")
            print(f"   Input: {len(chunk.split())} words, {len(chunk)} chars")
            if self.overlap_words > 0 and prev_translation:
                tail_words = len(get_overlap_context(prev_translation, self.overlap_words).split())
                print(f"   🔀 Context: last ~{tail_words} words of prev translation injected")

            try:
                translated = self.engine.translate(
                    chunk, self.tier, prev_translation, self.overlap_words)

                deva_flag = has_devanagari(translated)
                devanagari_flags.append(deva_flag)
                translations.append(translated)

                chunk_time = time.time() - chunk_start
                print(f"   Output: {len(translated)} chars")
                print(f"   {'⚠️  Devanagari detected — review this chunk!' if deva_flag else '✅ Script: Roman only'}")
                print(f"   ✅ Completed in {chunk_time:.1f}s")

                elapsed = time.time() - start_time
                avg = elapsed / i
                eta = (len(chunks) - i) * avg
                print(f"   📈 Progress: {i/len(chunks)*100:.1f}% | ETA: {eta/60:.1f}m")

            except Exception as e:
                print(f"   ❌ Error: {e}")
                translations.append(f"[TRANSLATION ERROR: {e}]")
                devanagari_flags.append(False)

        final_translation = "\n\n".join(translations)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        lang_code = self.target_lang.split('_')[0]
        output_file = self.output_dir / f"translation_{lang_code}_{timestamp}.txt"

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(final_translation)

        total_time = time.time() - start_time
        trans_chars = len(final_translation)
        deva_count = sum(devanagari_flags)

        print(f"\n{'=' * 70}")
        print(f"🎉 TRANSLATION COMPLETE!")
        print(f"{'=' * 70}")
        print(f"⏱️  Time: {total_time/60:.1f} minutes")
        print(f"📦 Chunks: {len(chunks)}")
        print(f"⚡ Avg/chunk: {total_time/len(chunks):.1f}s")
        print(f"📝 Input: {orig_chars:,} chars")
        print(f"📝 Output: {trans_chars:,} chars")
        print(f"📊 Ratio: {trans_chars/orig_chars:.2f}x")
        if deva_count > 0:
            print(f"⚠️  Devanagari found in {deva_count}/{len(chunks)} chunks — review flagged chunks")
        else:
            print(f"✅ All chunks passed Roman-only script check")
        print(f"💾 Output: {output_file}")
        print(f"{'=' * 70}")

        return str(output_file)


print("✅ Translation Engine v3 loaded and ready!")
print("   → Improved ADVANCED prompt (scene-aware, passive voice banned)")
print("   → Overlap context system (prev translation fed as context)")
print("   → Auto scene detection (ACTION / EMOTIONAL / PROPAGANDA / etc.)")
print("   → Devanagari post-processing check")
print("   → Temperature: 0.72 | Default chunk: 550 words")


🖥️ Using device: cuda
   GPU: Tesla T4
   Memory: 15.64 GB
✅ Translation Engine loaded and ready!


## 🌐 Step 5: Generate Translation
Run this cell to translate your uploaded file.

In [ ]:
# Create output directory
OUTPUT_DIR = "./translation_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize the generator based on provider
print("🚀 Initializing Translation Generator...")
print(f"   Provider: {SELECTED_PROVIDER}")
print(f"   Model: {SELECTED_MODEL}")
print(f"   Overlap: {OVERLAP_WORDS} words")

if SELECTED_PROVIDER == "ollama":
    generator = OllamaTranslationGenerator(
        model_name=SELECTED_MODEL,
        target_lang=TARGET_LANGUAGE,
        output_dir=OUTPUT_DIR,
        tier=TRANSLATION_TIER,
        chunk_size=CHUNK_SIZE,
        overlap_words=OVERLAP_WORDS
    )
else:
    generator = TranslationGenerator(
        model_name=SELECTED_MODEL,
        target_lang=TARGET_LANGUAGE,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
        tier=TRANSLATION_TIER,
        chunk_size=CHUNK_SIZE,
        hf_token=HF_TOKEN,
        overlap_words=OVERLAP_WORDS
    )

# Translate
print(f"\n🌐 Starting translation...")
OUTPUT_FILE = generator.translate_file(UPLOADED_FILE)
print(f"\n✅ Translation file generated: {OUTPUT_FILE}")


🚀 Initializing Translation Generator...
   Provider: ollama
   Model: gemma3:27b
📥 Initializing Ollama engine with model: gemma3:27b
   Target language: Hindi
⚠️ Model 'gemma3:27b' not found locally. Attempting to pull...
✅ Model 'gemma3:27b' pulled successfully!

🌐 Starting translation...

🌐 OLLAMA TRANSLATION GENERATOR
📄 Input: Animal_Farm_Chapter_05.txt
🦙 Model: gemma3:27b
🌐 Target: hin_Deva
🎯 Quality: ADVANCED

📊 Input: 17,995 chars, 3,271 words

📦 Creating chunks (650 words each)...
✅ Created 6 chunks

🎯 STARTING TRANSLATION


📄 Chunk 1/6
   Input: 650 words, 3530 chars
   Output: 3366 chars
   ✅ Completed in 467.7s
   📈 Progress: 16.7% | ETA: 39.0m

📄 Chunk 2/6
   Input: 650 words, 3560 chars
   Output: 3601 chars
   ✅ Completed in 461.1s
   📈 Progress: 33.3% | ETA: 31.0m

📄 Chunk 3/6
   Input: 650 words, 3640 chars
   Output: 3205 chars
   ✅ Completed in 428.4s
   📈 Progress: 50.0% | ETA: 22.6m

📄 Chunk 4/6
   Input: 650 words, 3553 chars
   Output: 3341 chars
   ✅ Completed in 

## 📖 Step 6: Download Translation


In [ ]:
# Download the translated file
from google.colab import files

print("📥 Downloading your translated file...")
files.download(OUTPUT_FILE)
print("✅ Download started! Check your browser's download folder.")

📥 Downloading your translated file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started! Check your browser's download folder.


## 💾 (Optional) Save to Google Drive
If you want to save the translation to your Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
import shutil

print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

# Create output folder in Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Translation_Output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Copy file to Drive
drive_output_path = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(OUTPUT_FILE))
shutil.copy(OUTPUT_FILE, drive_output_path)

print(f"\n✅ Translation saved to Google Drive:")
print(f"   📁 {drive_output_path}")

📂 Mounting Google Drive...


MessageError: Error: credential propagation was unsuccessful

---

## 📚 Quick Reference

### Supported Languages (NLLB Codes):
| Language | Code |
|----------|------|
| Hindi | `hin_Deva` |
| Bengali | `ben_Beng` |
| Tamil | `tam_Taml` |
| Telugu | `tel_Telu` |
| Marathi | `mar_Deva` |
| Gujarati | `guj_Gujr` |
| Spanish | `spa_Latn` |
| French | `fra_Latn` |
| German | `deu_Latn` |

### Recommended Models:
| Model | Best For | Speed |
|-------|----------|-------|
| `facebook/nllb-200-distilled-600M` | Fast multilingual | ⚡ Fast |
| `facebook/nllb-200-1.3B` | Better quality | 🔄 Medium |
| `ai4bharat/indictrans2-en-indic-1B` | Best EN→Hindi | 🔄 Medium |
| `google/madlad400-3b-mt` | Highest quality | 🐢 Slow |

### Quality Tiers:
- **BASIC**: Fast, good for simple texts
- **INTERMEDIATE**: Balanced quality and speed (recommended)
- **ADVANCED**: Best quality, preserves all nuances

### Tips:
- Use Colab GPU for faster translation
- For long texts, use smaller chunk sizes (200-300 words)
- NLLB models are best for multilingual translation